In [1]:
from langchain_openai import ChatOpenAI

In [34]:
model_name = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"
# Point to your local MLX server
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
    # Change "local-model" to the exact name shown in your server logs
    model=model_name, 
    temperature=0.0
)

In [17]:
import requests

# from langchain_classic import hub
# from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

In [18]:
my_stocks = {
    "tesla" : [],
    "google": [],
    "microsoft": [],
}

@tool
def get_current_stock(stock: str) -> list:
    """
    A function that returns the price history and current owner shares of the stock
    
    :param stock: the stock str value to retrieve from the storage
    :type stock: str

    :rtype: list of historical prices
    
    """
    return my_stocks[stock]

@tool
def update_price_history(stock: str, price: float) -> dict:
    """
    A function to update the stock price history of the given stock

    :param stock: The string value of the stock
    :type stock: str
    :param price: the float value of the price
    :type price: float

    :rtype: string value of price history update status

    """
    if stock not in my_stocks:
        my_stocks[stock] = []

    my_stocks[stock].append(price)
    return f"Update {stock}, total price history count  {len(my_stocks[stock])}"

search = DuckDuckGoSearchRun()
research_tools = [search]
accounting_tools = [get_current_stock, update_price_history]

In [20]:
prompt = """
You are a financial researcher. Find the last 7 days of closing prices for stocks.
"""

research_agent = create_react_agent(
    model=llm, 
    tools=research_tools, 
    name="researcher",
    prompt=prompt
)

prompt = """
You are an account. Record prices into the history and suggest buy/sell actions.
"""
accountant_agent = create_react_agent(
    model=llm,
    tools=accounting_tools,
    name="accountant",
    prompt=prompt
)

prompt = """
You are a team lead. First, ask the researcher to find the stock prices.
Second, pass those prices to the accountant to update the records.
when the accountant is done, respond with FINISH.
"""
workflow = create_supervisor(
    agents=[research_agent, accountant_agent],
    model=llm,
    prompt=prompt
)

# 3. Compile and Run with Readable Output
app = workflow.compile()

inputs = {"messages": [("user", "Update history for Tesla, Google, and Microsoft and give me a buy/sell tip.")]}

print("\n--- 🤖 Starting Agent Orchestration ---\n")

for event in app.stream(inputs, stream_mode="values"):
    # Get the latest message added to the state
    if "messages" in event:
        latest_msg = event["messages"][-1]
        
        # Determine the sender (Agent Name)
        sender = latest_msg.name if hasattr(latest_msg, 'name') and latest_msg.name else "System/User"
        content = latest_msg.content
        
        if content:
            print(f"[{sender.upper()}]:")
            print(f"{content}")
            print("-" * 30)

print("\n--- ✅ Task Completed ---")

/var/folders/h1/djwlcskd2sx4wv1g6s_thqpc0000gn/T/ipykernel_12144/2090546075.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(
/var/folders/h1/djwlcskd2sx4wv1g6s_thqpc0000gn/T/ipykernel_12144/2090546075.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  accountant_agent = create_react_agent(



--- 🤖 Starting Agent Orchestration ---

[SYSTEM/USER]:
Update history for Tesla, Google, and Microsoft and give me a buy/sell tip.
------------------------------
[TRANSFER_BACK_TO_SUPERVISOR]:
Successfully transferred back to supervisor
------------------------------
[SUPERVISOR]:
The update history and buy/sell advice are now transferred back to your supervisor. You can proceed with the next steps.
------------------------------

--- ✅ Task Completed ---


In [21]:
final_state = app.invoke(inputs)
print("\n--- FINAL RECOMMENDATION ---")
print(final_state["messages"][-1].content)


--- FINAL RECOMMENDATION ---
To provide a buy/sell tip for Tesla, Google, and Microsoft, we need the current stock prices. I will first update the history for these companies. Please wait for the price update.

<tools>
{"type": "function", "function": {"name": "fetch_stock_prices", "description": "Retrieve the latest stock prices for the past week", "parameters": {"properties": {}, "type": "object"}}}
</tools>


# state graphs

In [22]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
import operator

In [23]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    stock_data: str

In [ ]:
def research_node(state: AgentState):
    response = research_agent.invoke({"messages": state["messages"]})
    return {"messages": [response["messages"][-1]], "stock_data": response["messages"][-1].content}

def accountant_node(state: AgentState):
    prompt = f"Using this data: {state['stock_data']}, update the history and give a tip."
    response = accountant_agent.invoke({"messages": [HumanMessage(content=prompt)]})
    return {"messages": [response["messages"][-1]]}

In [32]:
workflow = StateGraph(AgentState)
workflow.add_node("researcher", research_node)
workflow.add_node("accountant", accountant_node)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "accountant")
workflow.add_edge("accountant", END)

app = workflow.compile()

In [ ]:
result = app.invoke(
    {"messages": [HumanMessage(content="Given the tools perform an internet search")]}
)

print("\n ----- Final Analysis -----")
print(result["messages"][-1].content)

state messages [HumanMessage(content='Update prices for Tesla...', additional_kwargs={}, response_metadata={})]

 ----- Final Analysis -----
I'm sorry, I can't provide current or historical prices for stocks like Tesla. However, I can help keep track of historical prices and provide advice based on past trends. Would you like me to start by retrieving the historical prices for a specific stock? Please specify the stock symbol you're interested in.
